# PDF Parsing — Built-in AI Functions

First half of the split built-in pipeline. Reads PDFs from the `rag_chatbot_dev.bronze.docs` volume and parses them with `ai_parse_document()`, persisting the structured `VARIANT` output to `rag_chatbot_dev.bronze.doc_parsed_builtin`.

A content-hash cache makes this step cheap to re-run: unchanged files keep their cached parse (parsing is model-backed and billed per invocation), only new or changed PDFs are re-parsed, and parses of files removed from the volume are purged. The companion notebook `PDF Chunking (built-in AI functions)` turns this table into chunks without any parsing cost.

In [0]:
# --- Configuration -------------------------------------------------------
CATALOG = "rag_chatbot_dev"
SCHEMA = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/docs"        # source volume containing the PDF files
PARSE_TABLE = f"{CATALOG}.{SCHEMA}.doc_parsed_builtin"   # parse cache (expensive, model-backed artifact)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PARSE_TABLE} (
    doc_path     STRING NOT NULL,
    doc_name     STRING,
    content_hash STRING,
    parsed       VARIANT,
    parsed_at    TIMESTAMP,
    CONSTRAINT doc_parsed_builtin_pk PRIMARY KEY (doc_path)
);
""")

# Every PDF currently in the volume, with a content hash
spark.sql(f"""
    WITH files AS (
        SELECT regexp_replace(path, '^(file://|dbfs:)', '') AS doc_path, content
        FROM READ_FILES('{VOLUME_PATH}/*.pdf', format => 'binaryFile')
    )
    SELECT
        doc_path,
        element_at(split(doc_path, '/'), size(split(doc_path, '/'))) AS doc_name,
        sha2(content, 256) AS content_hash,
        content
    FROM files
""").createOrReplaceTempView("volume_files")

spark.table(PARSE_TABLE).createOrReplaceTempView("parsed_cache")

# Cache misses: files that are new, or whose content changed since the last
# parse (LEFT ANTI JOIN keeps only rows with no exact doc_path + hash match)
spark.sql("""
    SELECT f.*
    FROM volume_files f
    LEFT ANTI JOIN parsed_cache c
        ON f.doc_path = c.doc_path AND f.content_hash = c.content_hash
""").createOrReplaceTempView("files_to_parse")

num_total = spark.table("volume_files").count()
num_to_parse = spark.table("files_to_parse").count()
print(f"{num_total} PDF(s) in volume — {num_to_parse} new or changed, {num_total - num_to_parse} served from parse cache")

In [0]:
# Cache-aware refresh: only new or changed files go through the model-backed
# ai_parse_document (billed per invocation). Unchanged files keep their
# cached parse; parses of files removed from the volume are purged.
changed_paths = [r.doc_path for r in spark.table("files_to_parse").select("doc_path").collect()]
all_paths = [r.doc_path for r in spark.table("volume_files").select("doc_path").collect()]

if changed_paths:
    # Drop superseded parses before inserting fresh ones (PK is doc_path)
    changed_sql = ", ".join(f"'{p.replace(chr(39), chr(39) * 2)}'" for p in changed_paths)
    spark.sql(f"DELETE FROM {PARSE_TABLE} WHERE doc_path IN ({changed_sql})")

    # Parse only the new or changed files — the billed, model-backed step
    spark.sql(f"""
        INSERT INTO {PARSE_TABLE}
        SELECT
            doc_path,
            doc_name,
            content_hash,
            ai_parse_document(content, MAP('version', '2.0')) AS parsed,
            current_timestamp() AS parsed_at
        FROM files_to_parse
    """)

if all_paths:
    # Purge parses of documents no longer in the volume
    all_sql = ", ".join(f"'{p.replace(chr(39), chr(39) * 2)}'" for p in all_paths)
    spark.sql(f"DELETE FROM {PARSE_TABLE} WHERE doc_path NOT IN ({all_sql})")
else:
    # Volume has no PDFs left: clear the cache so it stays consistent
    spark.sql(f"DELETE FROM {PARSE_TABLE} WHERE TRUE")

In [0]:
from pyspark.sql import functions as F

print(f"Total parses cached in {PARSE_TABLE}: {spark.table(PARSE_TABLE).count()}")

# Health check: error_status is a JSON null only when every page parsed cleanly
status = spark.sql(f"""
    SELECT
        COUNT(*) AS docs,
        SUM(CASE WHEN is_variant_null(parsed:error_status) THEN 1 ELSE 0 END) AS ok_docs
    FROM {PARSE_TABLE}
""").collect()[0]
print(f"{status['ok_docs']} of {status['docs']} cached parse(s) completed without page errors")

display(spark.table(PARSE_TABLE).select("doc_name", "parsed_at"))